# Using the indices API from RF-DETR

RF-DETR 1.12.1 converts the assignment returned by `batch_linear_assignment` into index pairs inside its matcher. This notebook puts that conversion beside `batch_linear_assignment_indices`, checks the pairs on a representative input, times the two paths, and compares the candidate public solver with an identified Git baseline.

Run all cells on an NVIDIA GPU with compute capability 8.0 or newer. The tutorial checks the actual Triton dispatch before taking timings. The revision benchmark runs many subprocesses and can take substantially longer than the tutorial cells.

## 0. Setup

Resolve the development branch once and use that full commit SHA for both the installed tutorial solver and the candidate benchmark checkout. A private clone keeps the revision checkouts separate from the notebook's installed module. If the kernel already loaded a different solver revision, restart it before running these cells.

In [ ]:
import json
import statistics
import sys
from importlib.metadata import distribution
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import torch
from IPython.display import display

In [ ]:
# Resolve one candidate for the install and the later benchmark subprocess.
REPO_URL = "https://github.com/Borda/torch-hungarian.git"
TUTORIAL_SESSION = TemporaryDirectory(prefix="torch-hungarian-rfdetr-")
REPO = Path(TUTORIAL_SESSION.name) / "repo"
!git clone -q --branch develop {REPO_URL} {REPO}
head_lines = !git -C {REPO} rev-parse HEAD
CANDIDATE_SHA = head_lines[0]
BASELINE_SHA = "881e5c3d6caa1f18853c6b5dbbe89a41ff949199"
print("Candidate revision:", CANDIDATE_SHA)
print("Post-release Git baseline:", BASELINE_SHA)

The installed package must come from the selected commit. Colab provides the runtime dependencies, so installation can leave the Torch and Triton wheels in place. The CUDA and import checks below fail early when the runtime cannot execute the example.

In [ ]:
assert torch.cuda.is_available(), "Choose a GPU runtime: Runtime → Change runtime type"
capability = torch.cuda.get_device_capability()
assert capability >= (8, 0), (
    f"{torch.cuda.get_device_name()} is sm_{capability[0]}{capability[1]}; Triton needs sm_80 or newer"
)
print(torch.cuda.get_device_name(), f"sm_{capability[0]}{capability[1]}", "torch", torch.__version__)

In [ ]:
# A live kernel keeps imported modules after pip changes the installation on disk.
if "torch_linear_assignment" in sys.modules:
    assert globals().get("LOADED_CANDIDATE_SHA") == CANDIDATE_SHA, "Restart the kernel to load the candidate revision"
else:
    !pip install -q --no-deps --force-reinstall "torch-hungarian @ git+{REPO_URL}@{CANDIDATE_SHA}"

installed = distribution("torch-hungarian")
provenance = json.loads(installed.read_text("direct_url.json") or "{}")
assert provenance.get("vcs_info", {}).get("commit_id") == CANDIDATE_SHA, (
    "Installed solver revision differs from candidate"
)

import torch_linear_assignment
from torch_linear_assignment import batch_linear_assignment, batch_linear_assignment_indices
from torch_linear_assignment.assignment import _uses_triton

assert (
    Path(torch_linear_assignment.__file__).resolve()
    == Path(installed.locate_file("torch_linear_assignment/__init__.py")).resolve()
), "Imported solver does not come from the verified installation"
LOADED_CANDIDATE_SHA = CANDIDATE_SHA

## 1. RF-DETR's solve today and its replacement

RF-DETR folds every `(layer, image, group)` problem of a training step into one batch of shape `[problems, group_width, targets]`: 300 queries per group, targets padded to the largest count in the batch. Below is that batch for 16 images with 13 query groups, so 208 problems, padded to 30 targets. The costs are random, which is enough to compare two code paths on the same input.

In [ ]:
# Match a representative RF-DETR batch so the two conversion paths see the same costs.
GROUP_WIDTH, MAX_TARGETS, IMAGES, GROUPS = 300, 30, 16, 13
generator = torch.Generator().manual_seed(0)
stacked = torch.rand((IMAGES * GROUPS, GROUP_WIDTH, MAX_TARGETS), generator=generator).cuda()
assert _uses_triton(stacked), "The example tensor must route to the Triton backend"
print(tuple(stacked.shape), stacked.dtype, stacked.device)

**Before.** This is RF-DETR 1.12.1's `rfdetr/models/_assignment.py::_solve_to_indices`. The solver returns one target per query (`-1` when a query is unmatched), so RF-DETR sorts a matched mask to list the matched queries in ascending order, gathers their targets, and copies both tensors to the host.

In [ ]:
def solve_to_indices_before(stacked: torch.Tensor, num_matches: int) -> tuple[torch.Tensor, torch.Tensor]:
    assignment = batch_linear_assignment(stacked)
    matched = assignment >= 0
    # Stable and descending puts every matched query first, in ascending row order.
    order = torch.argsort(matched.to(torch.int8), dim=1, descending=True, stable=True)
    rows = order[:, :num_matches]
    cols = assignment.gather(1, rows)
    pair = torch.stack((rows, cols)).cpu()
    return pair[0], pair[1]

**After.** `batch_linear_assignment_indices` returns the same pairs directly: `rows` sorted within each problem and `cols[b, i]` the target of query `rows[b, i]`, both `torch.long` on the input's device with shape `(problems, min(width, targets))`. The mask, the sort and the gather disappear, and `num_matches` is no longer needed because the shape already carries it.

In [ ]:
def solve_to_indices_after(stacked: torch.Tensor, num_matches: int) -> tuple[torch.Tensor, torch.Tensor]:
    rows, cols = batch_linear_assignment_indices(stacked)
    assert rows.shape[1] == num_matches
    pair = torch.stack((rows, cols)).cpu()
    return pair[0], pair[1]

## 2. The two return the same matching

Both functions run on the same batch. The shape is checked first because a wrong number of matches would otherwise surface as a confusing mismatch below.

In [ ]:
NUM_MATCHES = min(GROUP_WIDTH, MAX_TARGETS)
rows_before, cols_before = solve_to_indices_before(stacked, NUM_MATCHES)
rows_after, cols_after = solve_to_indices_after(stacked, NUM_MATCHES)
assert rows_before.shape == rows_after.shape == (IMAGES * GROUPS, NUM_MATCHES)

In [ ]:
# Exact equality, not closeness: both paths read the same solver output, and RF-DETR's parity tests require identical index pairs on continuous costs.
assert torch.equal(rows_before, rows_after)
assert torch.equal(cols_before, cols_after)
print("identical (rows, cols) for", rows_after.shape[0], "problems")

On the Triton path, `validate="off"` skips the numeric and feasibility readbacks and allows CUDA graph capture after warm-up. Use it when the costs are known to be finite and feasible. The next cell checks that both modes agree on the valid example.

In [ ]:
rows_off, cols_off = batch_linear_assignment_indices(stacked, validate="off")
assert torch.equal(rows_off.cpu(), rows_after)
assert torch.equal(cols_off.cpu(), cols_after)

## 3. Timing at RF-DETR's batch sizes

A training step solves 52, 208 or 624 problems depending on batch size and decoder layers. Each call is timed after a warm-up with a device synchronize around it, so the number includes the host copy RF-DETR does, which is the cost a training step actually pays.

In [ ]:
def median_ms(function, stacked, num_matches, repeats=30, warmup=5):
    for _ in range(warmup):
        function(stacked, num_matches)
    samples = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        function(stacked, num_matches)
        end.record()
        torch.cuda.synchronize()
        samples.append(start.elapsed_time(end))
    return statistics.median(samples)

In [ ]:
# Check dispatch for each timed shape so a host fallback cannot enter the GPU table.
rows = []
for problems in (52, 208, 624):
    batch = torch.rand((problems, GROUP_WIDTH, MAX_TARGETS), generator=generator).cuda()
    assert _uses_triton(batch), f"Expected the Triton backend for batch size {problems}"
    before = median_ms(solve_to_indices_before, batch, NUM_MATCHES)
    after = median_ms(solve_to_indices_after, batch, NUM_MATCHES)
    rows.append(
        {
            "problems": problems,
            "before_ms": round(before, 2),
            "after_ms": round(after, 2),
            "after / before": round(after / before, 2),
        }
    )
display(pd.DataFrame(rows))

Read the last column as a ratio of times: below `1.0` means the new code is faster. The two paths share the solver, so the gain is limited to the conversion RF-DETR no longer does. Treat a ratio near `1.0` as expected, not as a failure.

## 4. Compare the candidate with the Git baseline

The earlier timings isolate the conversion change while both paths use the same solver. This section compares the public call across two Git revisions. The baseline `881e5c3` contains the comparable benchmark harness and post-release changes, so it represents that Git revision. The published RF-DETR dependency would need a separate measurement. Both revisions run the same preset on the same GPU with three fresh-process rounds.

In [ ]:
def run_rfdetr_preset(revision: str) -> Path:
    """Benchmark the requested revision using its own script and solver."""
    !git -C {REPO} checkout -q --detach {revision}
    output = REPO.parent / f"benchmark-{revision[:12]}.jsonl"
    # A leftover file from an earlier failed run must not pass for this run's result.
    output.unlink(missing_ok=True)
    !cd {REPO} && env -u TLA_BENCHMARK_SOURCE_PATH -u TLA_BENCHMARK_GIT_REVISION PYTHONPATH={REPO}/src python tests/benchmark.py --preset rfdetr --backends scipy,public_cuda,triton --dtypes float32 --process-rounds 3 --expect-implementation triton --output {output}
    return output

In [ ]:
# Keep separate files for the two immutable revisions so their results stay identifiable.
baseline_records = run_rfdetr_preset(BASELINE_SHA)

In [ ]:
candidate_records = run_rfdetr_preset(CANDIDATE_SHA)

Read the full-validation public CUDA warm median for each shape. The historical harness measures the complete call. Compare the raw ratio with 1.05 and round the value for display, so a slowdown just above 5% remains visible.

In [ ]:
def warm_medians(path: Path, revision: str) -> dict[tuple[int, int, int], float]:
    """Read the warm medians from complete public Triton records for a revision."""
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    medians = {}
    for record in records:
        if record.get("record_type") != "case" or record.get("backend") != "public_cuda":
            continue
        source = record["run_metadata"]["source"]
        stats = record["process_statistics"]
        assert source["benchmark_git_revision"] == revision and source["benchmark_git_dirty"] == "clean"
        assert (
            Path(source["assignment_module_origin"]) == (REPO / "src/torch_linear_assignment/assignment.py").resolve()
        )
        assert record["implementation"] == "triton" and record["parity"]["exact"]
        assert record["validation_mode"] == "full" and stats["status"] == "measured" and stats["rounds"] == [0, 1, 2]
        shape = record["shape"]
        medians[(shape["batch"], shape["workers"], shape["tasks"])] = stats["warm_median_ms"]["median"]
    return medians

In [ ]:
baseline = warm_medians(baseline_records, BASELINE_SHA)
candidate = warm_medians(candidate_records, CANDIDATE_SHA)
expected_shapes = {(batch, 300, tasks) for batch in (52, 208, 624) for tasks in (1, 2, 5, 10, 20, 30)}
assert baseline.keys() == candidate.keys() == expected_shapes, "Both revisions must measure every RF-DETR shape"

table = pd.DataFrame(
    [
        {
            "batch": key[0],
            "tasks": key[2],
            "baseline_ms": baseline[key],
            "candidate_ms": candidate[key],
            "candidate / baseline": candidate[key] / baseline[key],
        }
        for key in sorted(baseline)
    ]
)
table["slower than 1.05x"] = table["candidate / baseline"] > 1.05
display(table.round({"baseline_ms": 2, "candidate_ms": 2, "candidate / baseline": 2}))

A row marked `True` means the candidate is more than 5% slower than the Git baseline for that shape on this GPU. Record both full revision SHAs and keep the JSONL alongside the result.

## 5. What RF-DETR should change

1. In `rfdetr/models/_assignment.py`, replace the body of `_solve_to_indices` with the "after" version from section 1: call `batch_linear_assignment_indices(stacked)`, stack `rows` and `cols`, and copy them to the host once.
2. In RF-DETR's `train` extra, relax the exact pin `torch-hungarian==0.1.0rc0` to `torch-hungarian>=0.2.0`. Until RF-DETR does, its users get none of the kernel or validation changes.
3. Keep the default `validate="full"` unless the matcher can guarantee finite, feasible costs. With `validate="off"`, warm up the inputs that route to Triton before CUDA graph capture.